# CSV analysis (offline — no Bloomberg)

1. Pon `data_11_22.csv` en la raíz del repo
2. Run All
3. Outputs → `exports/csv_analysis/`

**Columnas esperadas:** `USOSFR1 Curncy` … `USOSFR50 Curncy`, `USSFCT05/10/30 Curncy`, `ESA Index`, `CO1 Comdty` (+ columna de fecha/time).

Curvas / flies = mismas fórmulas que el screener (`CURVES` / `FLIES` en `curve_config.py`), en **bp**.


In [ ]:
import importlib, logging, os, sys
from pathlib import Path
from IPython.display import display, Image
import pandas as pd
import matplotlib.pyplot as plt

logging.disable(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

_DZ = Path("/v/region/na/appl/datazone/notebooks2/data/users/arraipau/fwd_screener_v1")
REPO = Path.cwd() if (Path.cwd() / "lib" / "csv_analysis.py").exists() else _DZ
if not (REPO / "lib" / "csv_analysis.py").exists():
    raise FileNotFoundError("No lib/csv_analysis.py — git pull origin main")
os.chdir(REPO)
sys.path = [str(REPO)] + [p for p in sys.path if str(Path(p).resolve()) != str(REPO.resolve())]
print("REPO:", REPO.resolve())


In [ ]:
CSV_PATH = "data_11_22.csv"   # raíz del repo
OUT_DIR = "exports/csv_analysis"
FFILL = True

import lib.csv_analysis as csv_analysis
importlib.reload(csv_analysis)
print("csv exists:", Path(CSV_PATH).exists(), Path(CSV_PATH).resolve())


## 1) Load + clean


In [ ]:
raw = csv_analysis.load_csv(CSV_PATH, ffill=False)
print("raw shape:", raw.shape)
print("raw columns (first 25):", list(raw.columns)[:25])
raw.tail(3)


In [ ]:
cleaned, report = csv_analysis.clean_panel(raw)
if FFILL:
    cleaned = cleaned.ffill()
print("report:", {k: report[k] for k in ("start", "asof", "n_rows", "tenors", "spreads", "extras")})
cleaned.tail(5)


## 2) Curves / flies + analysis panel

Rates (%) + curves/flies (bp) + USSFCT + ESA/CO1 → `exports/csv_analysis/analysis_panel.csv`


In [ ]:
panel = csv_analysis.build_analysis_panel(cleaned)
panel_path = csv_analysis.save_panel(panel, out_dir=OUT_DIR)
struct_cols = [c for c in panel.columns if not str(c).isdigit() and not str(c).startswith("USSFCT") and c not in ("ESA", "CO1")]
print("structures:", struct_cols)
panel[struct_cols].tail(5) if struct_cols else panel.tail(5)


## 3) Metrics + charts

Moves 1d/5d/21d, curve snapshot, time series, rolling z, corr heatmap, extras vs 10y.


In [ ]:
paths = csv_analysis.run_all_metrics(panel, out_dir=OUT_DIR)
moves = pd.read_csv(Path(OUT_DIR) / "recent_moves.csv")
print("wrote:", sorted(Path(OUT_DIR).glob("*")))
display(moves.round(3))


In [ ]:
# show all PNGs
for png in sorted(Path(OUT_DIR).glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))


## 4) Correlaciones (ρ + percentil en el heatmap)

- **`corr_heatmap_rho_pct.png`** — color = ρ, texto = `ρ` + `pXX` (percentil)
- Evolución rolling ρ: **10y** y **USSFCT30** vs **ESA / CO1**
- Scorecard + scatter ρ(10y, ESA) vs ρ(10y, CO1)


In [ ]:
CORR_WINDOW = 50
PCT_LOOKBACK = 200

corr_out = csv_analysis.run_corr_analysis(
    panel, out_dir=OUT_DIR, corr_window=CORR_WINDOW, pct_lookback=PCT_LOOKBACK,
)
print("corr files:", sorted(k for k in corr_out["paths"]))
if corr_out.get("scorecard") is not None and len(corr_out["scorecard"]):
    print("\n=== pair scorecard (ρ last + percentile) ===")
    display(corr_out["scorecard"].round(2))
print("\nlevel corr:")
display(corr_out["level"].round(3))
print("percentile (100 = max histórico):")
display(corr_out["percentile"].round(1))


In [ ]:
# heatmaps (ρ+percentil primero) + evolución + scatter
for name in (
    "corr_heatmap_rho_pct.png",
    "corr_percentile_heatmap.png",
    "corr_level_heatmap.png",
    "corr_evol_10y_vs_extras.png",
    "corr_evol_ussfct30_vs_extras.png",
    "corr_scatter_10y_esa_vs_co1.png",
):
    fp = Path(OUT_DIR) / name
    if fp.exists():
        print(name)
        display(Image(filename=str(fp)))
    else:
        print("missing", name)


## One-shot (opcional)

Equivalente a Run All de análisis:


In [ ]:
# panel, report, paths = csv_analysis.run_pipeline(CSV_PATH, out_dir=OUT_DIR, ffill=FFILL)
# panel.tail()
